# Calibration diagnostics

Goal: diagnose why the XGBRanker + per-race softmax model produces very negative ROI and systematically assigns positive EV to underdogs / negative EV to favorites.

All analysis is on the **validation** split. Test is held back.

## Diagnosis (TL;DR)

The softmax-over-ranker-scores pipeline produces probabilities that are **too flat**. Per-race entropy is +0.22 nats above market on average (1.87 vs 1.65). The miscalibration increases monotonically with field size (entropy gap 0.17 in ≤6-horse fields, 0.31 in 11+). A scalar temperature sweep finds log-loss minimized at **T ≈ 0.5**, dropping log-loss from 1.696 to 1.613 — within 0.02 of the market's 1.591. Single-parameter temperature scaling is the obvious next step; if post-scaling log-loss still trails the market, move on to per-race isotonic scaling or a log-linear blend with market_prob.

The symptom on betting: every horse at odds 20+ is flagged positive-EV (9,950 / 9,950) with flat-stake ROI of **-45%**; 10–20 bucket ROI is **-26%**. Favorites are assigned 27% probability when market says 38% — so they're systematically graded negative-EV even when they're the right bet.

In [1]:
from pathlib import Path

import numpy as np
import polars as pl
import shap

from api.predict import load_model
from model import diagnostics as diag
from model.evaluate import _log_loss_winner, _top1_accuracy
from model.features import build_training_df, split_by_race
from model.train import prepare_df

In [2]:
bundle = load_model(model_dir=Path("../model/artifacts"))
model = bundle["model"]
features = bundle["features"]

df = build_training_df(processed_dir=Path("../data/processed"))
train_df, val_df, test_df = split_by_race(df)
print(f"val: {val_df.shape[0]:,} rows, {val_df['race_id'].n_unique():,} races")

val: 46,888 rows, 6,350 races


In [3]:
val = diag.enrich(val_df, model, features)

# sanity checks: probs sum to 1 per race, log-loss and top-1 match evaluate.py
prob_sums = val.group_by("race_id").agg(pl.col("model_prob").sum().alias("s"))["s"].to_numpy()
assert np.allclose(prob_sums, 1.0, atol=1e-6), f"probs don't sum to 1 (range {prob_sums.min()}-{prob_sums.max()})"
print(f"model  log-loss: {_log_loss_winner(val, 'model_prob'):.4f}   top-1 acc: {_top1_accuracy(val, 'model_prob'):.4f}")
print(f"market log-loss: {_log_loss_winner(val, 'market_prob'):.4f}   top-1 acc: {_top1_accuracy(val, 'market_prob'):.4f}")

model  log-loss: 1.6955   top-1 acc: 0.3872
market log-loss: 1.5910   top-1 acc: 0.3948


## 1. Is the model's distribution flatter than the market's?

If softmax-over-ranker-scores is under-sharpened, per-race entropy will exceed the market's, favorites will get less mass than the market assigns, and the `model_prob` vs `market_prob` scatter will sit below the diagonal for high-prob horses / above it for low-prob horses.

In [4]:
diag.entropy_summary(val)

model_entropy_mean,market_entropy_mean,delta_mean,n_races
f64,f64,f64,i64
1.868737,1.652114,0.216624,6335


In [5]:
diag.plot_entropy_hist(val).show()

In [6]:
diag.plot_favorite_prob_hist(val).show()

In [7]:
diag.plot_prob_scatter(val, sample=10_000).show()

## 2. Reliability diagrams

Horses are binned by predicted probability; each bin plots mean predicted vs empirical win rate with a Wilson 95% CI. Market is overlaid as a reference.

In [8]:
rel_model = diag.reliability_table(val, "model_prob")
rel_market = diag.reliability_table(val, "market_prob")
print(f"model  Brier: {diag.brier_score(val, 'model_prob'):.4f}")
print(f"market Brier: {diag.brier_score(val, 'market_prob'):.4f}")
rel_model

model  Brier: 0.1040
market Brier: 0.0995


bin,mean_pred,empirical_rate,count,wins,ci_lo,ci_hi
cat,f32,f64,u32,i64,f64,f64
"""[2%, 5%)""",0.046141,0.004831,2277,11,0.0027,0.00863
"""[5%, 10%)""",0.076141,0.038446,15060,579,0.035491,0.041637
"""[10%, 15%)""",0.124532,0.107862,14259,1538,0.102875,0.113059
"""[15%, 20%)""",0.17146,0.191794,8384,1608,0.183508,0.200362
"""[20%, 30%)""",0.240543,0.325304,5106,1661,0.312589,0.338281
"""[30%, 50%)""",0.369069,0.541219,1674,906,0.517281,0.564968
"""[50%, 100%)""",0.545462,0.727273,44,32,0.581508,0.836538


In [9]:
rel_market

bin,mean_pred,empirical_rate,count,wins,ci_lo,ci_hi
cat,f64,f64,u32,i64,f64,f64
"""[0%, 2%)""",0.0143,0.005861,3583,21,0.003837,0.008944
"""[2%, 5%)""",0.034221,0.028776,9487,273,0.025598,0.032336
"""[5%, 10%)""",0.073259,0.070128,10823,759,0.065468,0.075094
"""[10%, 15%)""",0.12379,0.119451,7141,853,0.112133,0.127179
"""[15%, 20%)""",0.17329,0.165024,5072,837,0.155062,0.175492
"""[20%, 30%)""",0.244082,0.243965,5841,1425,0.233122,0.255145
"""[30%, 50%)""",0.374484,0.407417,4072,1659,0.392419,0.422589
"""[50%, 100%)""",0.576737,0.647134,785,508,0.613062,0.679772


In [10]:
diag.plot_reliability(rel_model, rel_market).show()

## 3. Odds-bucket breakdown

Buckets horses by `dollar_odds`. If the symptom is real, favorites (`<2`, `2-5`) will have model_prob below market_prob and negative mean EV; longshots (`20+`) will have model_prob above market_prob and positive mean EV. The rightmost columns show hit rate and ROI among horses the `ev_threshold > 0` rule would have bet on.

In [11]:
diag.odds_bucket_table(val)

odds_bucket,n,mean_model_prob,mean_market_prob,win_rate,mean_ev,n_positive_ev,pos_ev_hit_rate,pos_ev_roi
cat,u32,f32,f64,f64,f64,u32,f64,f64
"""<2""",6247,0.272405,0.378373,0.407556,-0.410083,8,0.5,0.29375
"""2-5""",11317,0.167333,0.191632,0.189184,-0.269397,302,0.195364,-0.012748
"""5-10""",10378,0.124946,0.102821,0.098478,0.018102,5056,0.088212,-0.212184
"""10-20""",8912,0.092269,0.05574,0.050382,0.388747,8702,0.050448,-0.264501
"""20+""",9950,0.062369,0.023997,0.017789,1.390451,9950,0.017789,-0.454007


## 4. Temperature probe

Sweep a scalar `T` and compute log-loss of `softmax(score / T)` per race. If `T < 1` minimizes log-loss, softmax is too flat (need to sharpen). If the minimum is near `T = 1`, the problem is not a simple temperature fix.

In [12]:
temps = np.concatenate([np.linspace(0.1, 1.0, 19), np.linspace(1.1, 5.0, 20)])
sweep = diag.temperature_sweep(val, temps)
market_ll = _log_loss_winner(val, "market_prob")
diag.plot_temperature_sweep(sweep, baseline=market_ll).show()
sweep.sort("log_loss").head(5)

T,log_loss
f64,f64
0.5,1.613441
0.55,1.615767
0.45,1.617157
0.6,1.621679
0.65,1.629664


## 5. SHAP: favorites vs longshots

Cross-check whether the model uses features differently for favorites vs longshots. If `morning_line_odds_float` / `dollar_odds_plus_noise` have dramatically different mean |SHAP| between the two subsets, the model may be underweighting odds information in one regime.

In [13]:
explainer = shap.TreeExplainer(model)
X_val, _, _ = prepare_df(val_df, features)
shap_values = explainer.shap_values(X_val)
shap_values.shape

(46888, 67)

In [14]:
# align shap rows to val_df (prepare_df sorts by race_id)
val_sorted = val_df.sort("race_id")
is_fav = val_sorted["dollar_odds"].to_numpy() < 2.0
is_long = val_sorted["dollar_odds"].to_numpy() >= 20.0

fav_shap = np.abs(shap_values[is_fav]).mean(axis=0)
long_shap = np.abs(shap_values[is_long]).mean(axis=0)

shap_cmp = (
    pl.DataFrame({
        "feature": features,
        "mean_abs_shap_favorites": fav_shap,
        "mean_abs_shap_longshots": long_shap,
        "delta_fav_minus_long": fav_shap - long_shap,
    })
    .sort("mean_abs_shap_favorites", descending=True)
)
print(f"favorites n={is_fav.sum():,}   longshots n={is_long.sum():,}")
shap_cmp.head(15)

favorites n=5,846   longshots n=10,021


feature,mean_abs_shap_favorites,mean_abs_shap_longshots,delta_fav_minus_long
str,f32,f32,f32
"""dollar_odds_plus_noise""",0.408789,0.774267,-0.365478
"""dollar_odds_plus_noise_rank""",0.046377,0.074508,-0.028131
"""field_size""",0.037694,0.02339,0.014304
"""ml_odds_rank""",0.010766,0.008492,0.002274
"""speed_fig_minus_field_avg_L1""",0.008763,0.004054,0.004708
…,…,…,…
"""entry_to_race_class_ratio""",0.002892,0.004555,-0.001664
"""class_rating_L3""",0.002091,0.000605,0.001485
"""best_workout_rank_pct""",0.001615,0.00713,-0.005514


## 6. Field-size interaction

Softmax flatness compounds with field size: residual mass is spread over more runners, so each longshot's prob is inflated more in a 12-horse field than a 6-horse field. If the hypothesis is right, `entropy_gap` (model - market) and `log_loss_gap` should grow with field size, and the `favorite_gap` should become more negative (model under-assigns the favorite by more) in larger fields.

In [15]:
diag.field_size_bucket_table(val)

field_bucket,n_races,entropy_gap,favorite_gap,log_loss_gap
cat,u32,f64,f64,f64
"""<=6""",2119,0.165076,-0.089423,0.093451
"""7-8""",2433,0.218212,-0.102696,0.097031
"""9-10""",1444,0.268303,-0.115986,0.122079
"""11+""",339,0.307302,-0.116363,0.152686


In [16]:
# reliability stratified into small (<=8) vs large (>=11) fields
small = val.filter(pl.col("field_size") <= 8)
large = val.filter(pl.col("field_size") >= 11)
print(f"small fields: {small['race_id'].n_unique():,} races   large fields: {large['race_id'].n_unique():,} races")

rel_small = diag.reliability_table(small, "model_prob")
rel_large = diag.reliability_table(large, "model_prob")
fig = diag.plot_reliability(rel_small, rel_large, title="Reliability: small (blue) vs large (orange) fields")
fig.data[1].name = "small fields (<=8)"
fig.data[2].name = "large fields (>=11)"
fig.show()

small fields: 4,552 races   large fields: 339 races


## Diagnosis summary

- **Distribution flatness (sec 1):** confirmed. Model per-race entropy 1.87 vs market 1.65 (+0.22 nats). Favorite probability histogram shifted left of market. Scatter sits well below y=x for high-prob horses and above it for low-prob horses.
- **Reliability (sec 2):** severely miscalibrated on the low end. Model predicts 4.6% for horses that actually win 0.5% (≈10× overestimate) and 7.6% for horses that win 3.8% (≈2× overestimate). Market reliability hugs the diagonal. Brier: model 0.104 vs market 0.099.
- **Odds-bucket symptom (sec 3):** direct confirmation. Favorites (`<2`): model prob 27% vs market 38%, mean EV -0.41. Longshots (`20+`): model prob 6.2% vs market 2.4%, mean EV +1.39. `ev_threshold > 0` rule ROI worsens monotonically as odds rise: -21% (5-10), -26% (10-20), -45% (20+).
- **Temperature probe (sec 4):** minimum at **T ≈ 0.5**, log-loss 1.613 (from 1.696 baseline, market 1.591). Single scalar temperature recovers most of the gap to the market — strong evidence the dominant problem is softmax temperature, not a deeper model issue.
- **SHAP favorites vs longshots (sec 5):** no dramatic asymmetry; `morning_line_odds_float` is dominant for both subsets. Not the root cause.
- **Field-size interaction (sec 6):** confirmed. Entropy gap, favorite gap, and log-loss gap all grow monotonically with field size — exactly the pattern expected if softmax is spreading residual mass across more runners in larger fields.

**Recommended next step:** implement temperature scaling. Fit `T` on validation (current optimum ≈ 0.5), persist it alongside the model bundle, and apply `score / T` inside `_per_race_softmax` at inference time. Re-evaluate log-loss, reliability, and ROI. If log-loss still trails the market by > 0.01 nats after scaling, escalate to per-race isotonic regression or a log-linear blend with `market_prob`.